<a href="https://colab.research.google.com/github/Vdivyajaswanthi123/ai-mentor-portfolio/blob/main/day10_sprint5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q "numpy<2.0"
!pip install -q --upgrade crewai litellm google-generativeai google-genai chromadb sentence-transformers crewai-tools

In [ ]:
import os, getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter Gemini API Key: ")

print("API key set")

In [ ]:
!pip uninstall -y numpy
!pip install numpy --upgrade --force-reinstall

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

from chromadb import PersistentClient
from sentence_transformers import SentenceTransformer

import json
import pathlib

In [ ]:
llm = "gemini/gemini-2.5-flash"

print(llm)

In [ ]:
from crewai.tools import tool
from chromadb import PersistentClient
from sentence_transformers import SentenceTransformer


client_db = PersistentClient(path="./chroma_db")

col = client_db.get_or_create_collection(
    name="placement_kb"
)

embed = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)


@tool
def rag_search(query: str) -> str:
    """
    Search placement knowledge base
    """

    qv = embed.encode([query]).tolist()

    results = col.query(
        query_embeddings=qv,
        n_results=4
    )

    docs = results["documents"][0]

    return "\n---\n".join(docs)

In [ ]:
researcher = Agent(
    role="Placement Researcher",

    goal="Research company placement requirements for students",

    backstory="You prepare factual placement research notes using RAG search.",

    llm=llm,

    tools=[rag_search],

    verbose=True,

    allow_delegation=False,
)

In [ ]:
interviewer = Agent(
    role="Mock Interviewer",

    goal="Generate placement interview questions",

    backstory="You generate technical and HR interview questions.",

    llm=llm,

    verbose=True,

    allow_delegation=False,
)

In [ ]:
coach = Agent(
    role="Answer Coach",

    goal="Create strong sample answers and preparation guidance",

    backstory="You coach students with strong placement answers.",

    llm=llm,

    verbose=True,

    allow_delegation=False,
)

In [ ]:
tracker = Agent(
    role="Progress Tracker",

    goal="Create structured student progress summaries",

    backstory="You generate JSON-style student progress summaries.",

    llm=llm,

    verbose=True,

    allow_delegation=False,
)

In [ ]:
profiles = [
    {
        "student_id": "S001",
        "name": "Ravi Kumar",
        "branch": "CSE",
        "cgpa": 7.8,
        "skills": ["Python", "Java", "SQL"],
        "target_company": "TCS Digital"
    },

    {
        "student_id": "S002",
        "name": "Sneha Reddy",
        "branch": "ECE",
        "cgpa": 8.1,
        "skills": ["Python", "DBMS"],
        "target_company": "Cognizant"
    },

    {
        "student_id": "S003",
        "name": "Arun Pillai",
        "branch": "IT",
        "cgpa": 8.5,
        "skills": ["Java", "DSA", "OOPs"],
        "target_company": "Amazon"
    }
]

In [ ]:
def build_tasks(profile):

    research = Task(
        description=(
            f"Research placement preparation details for "
            f"{profile['target_company']} "
            f"for a {profile['branch']} student."
        ),

        agent=researcher,

        expected_output="Short bullet research notes."
    )

    interview = Task(
        description=(
            f"Generate EXACTLY 10 mock interview questions "
            f"for {profile['name']} targeting "
            f"{profile['target_company']}."
        ),

        agent=interviewer,

        expected_output="Exactly 10 numbered interview questions."
    )

    coaching = Task(
        description=(
            f"Pick question 3 and create strong sample answer "
            f"for {profile['name']}."
        ),

        agent=coach,

        expected_output="One strong answer and 3 preparation tips."
    )

    tracking = Task(
        description=(
            f"Create JSON-style progress summary "
            f"for {profile['student_id']}."
        ),

        agent=tracker,

        expected_output="Valid JSON-style summary."
    )

    return [research, interview, coaching, tracking]

In [ ]:
p = profiles[0]

crew = Crew(
    agents=[
        researcher,
        interviewer,
        coach,
        tracker
    ],

    tasks=build_tasks(p),

    process=Process.sequential,

    verbose=True,
)

In [ ]:
result = await crew.kickoff_async()

print("\n=== FINAL OUTPUT ===\n")

print(result)

In [ ]:
import os
import getpass

# remove old Gemini key
os.environ.pop("GEMINI_API_KEY", None)

# enter new Gemini key
os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter Gemini API Key: ")

In [ ]:
import os

print(os.environ["GEMINI_API_KEY"][:10])

In [ ]:
import asyncio

transcripts = []

for p in profiles[:1]:
    print(f"Running for {p['name']} → {p['target_company']}")

    result = await crew.kickoff_async()

    transcripts.append({
        "student": p["name"],
        "target": p["target_company"],
        "final_output": str(result)
    })

    await asyncio.sleep(30)   # optional pause between runs

In [ ]:
pathlib.Path(
    "day10_sprint5_transcripts.json"
).write_text(
    json.dumps(transcripts, indent=2)
)

print("Saved transcripts successfully")

In [ ]:
md = "# Day10 Sprint5 Report\n\n"

for t in transcripts:

    md += f"## {t['student']} → {t['target']}\n\n"

    md += t["final_output"]

    md += "\n\n---\n\n"

pathlib.Path(
    "day10_sprint5_report.md"
).write_text(md)

print("Markdown report created")

In [ ]:
from google.colab import files

files.download("day10_sprint5_transcripts.json")

files.download("day10_sprint5_report.md")